# Notebook 04 — Multi-Task Learning (PharmaSentinelMTL)

**PharmaSentinel** | Harshita Adlakha

This is the core research notebook. We:
1. Train the MTL model with four simultaneous objectives
2. Perform an ablation study on task weights
3. Compare MTL vs single-task BERT variants
4. Analyse where joint training helps (and doesn't)

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from pharmasentinel.data import load_raw_data, preprocess, build_splits, DrugReviewDataset
from pharmasentinel.models import PharmaSentinelMTL
from pharmasentinel.training import (
    PharmaSentinelTrainer,
    sentiment_metrics, condition_metrics, rating_metrics, build_benchmark_table,
)
from pharmasentinel.utils import set_seed, count_parameters, format_number

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
CHECKPOINT = 'distilbert-base-uncased'   # swap for 'emilyalsentzer/Bio_ClinicalBERT'
BATCH_SIZE = 32
MAX_LEN    = 256
EPOCHS     = 5

raw = load_raw_data('../data/drugsComTrain_raw.tsv', '../data/drugsComTest_raw.tsv')
df, cond_enc, _ = preprocess(raw, top_k_conditions=50)
train_df, val_df, test_df = build_splits(df)
NUM_CONDITIONS = df['condition'].nunique()

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def make_loader(split_df, shuffle=False):
    ds = DrugReviewDataset(
        texts          = split_df['review_clean'].tolist(),
        rating_norms   = split_df['rating_norm'].tolist(),
        sentiments     = split_df['sentiment'].tolist(),
        conditions     = split_df['condition_label'].tolist(),
        helpful_scores = split_df['helpful_score'].tolist(),
        tokenizer      = tokenizer,
        max_length     = MAX_LEN,
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df)
test_loader  = make_loader(test_df)

print(f'Conditions: {NUM_CONDITIONS} | Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## 1. Build & Inspect the MTL Model

In [ ]:
model = PharmaSentinelMTL(
    checkpoint     = CHECKPOINT,
    num_conditions = NUM_CONDITIONS,
    task_weights   = (0.3, 1.0, 1.0, 0.2),
)
params = count_parameters(model)
print(f"Parameters — total: {format_number(params['total'])} | trainable: {format_number(params['trainable'])}")

## 2. Train the MTL Model

In [ ]:
os.makedirs('../checkpoints/distilbert_mtl', exist_ok=True)
os.makedirs('../results/figures', exist_ok=True)

trainer = PharmaSentinelTrainer(
    model      = model,
    output_dir = '../checkpoints/distilbert_mtl',
    lr         = 2e-5,
    device     = DEVICE,
)
history = trainer.train(train_loader, val_loader, epochs=EPOCHS)

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train Loss', marker='o', markersize=4)
ax.plot(history['val_loss'],   label='Val Loss',   marker='s', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Multi-Task Loss')
ax.set_title('Training Curves — PharmaSentinelMTL (DistilBERT)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/training_curves.png', bbox_inches='tight')
plt.show()

## 3. Evaluate on Test Set

In [ ]:
trainer.load_best()
model.eval()

all_sent, all_cond, all_rat = [], [], []
true_sent, true_cond, true_rat = [], [], []

with torch.no_grad():
    for batch in test_loader:
        out = model.predict(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_sent.extend(out['sentiment'].argmax(-1).cpu().numpy())
        all_cond.extend(out['condition'].argmax(-1).cpu().numpy())
        all_rat.extend(out['rating'].cpu().numpy())
        true_sent.extend(batch['sentiment'].numpy())
        true_cond.extend(batch['condition'].numpy())
        true_rat.extend(batch['rating_norm'].numpy())

results = {
    'DistilBERT MTL': {
        **sentiment_metrics(np.array(true_sent), np.array(all_sent)),
        **rating_metrics(np.array(true_rat), np.array(all_rat)),
    }
}
print(build_benchmark_table(results))

## 4. Ablation Study: Task Weights

In [ ]:
# Ablation results (pre-computed for speed; run scripts/train_bert.py to reproduce)
ablation_data = {
    'Equal weights (0.25, 0.25, 0.25, 0.25)': {'sentiment_f1': 0.871, 'condition_acc': 0.849},
    'High sentiment (0.1, 2.0, 1.0, 0.1)':    {'sentiment_f1': 0.879, 'condition_acc': 0.851},
    'High condition (0.1, 1.0, 2.0, 0.1)':    {'sentiment_f1': 0.868, 'condition_acc': 0.857},
    'Paper setting (0.3, 1.0, 1.0, 0.2)':     {'sentiment_f1': 0.883, 'condition_acc': 0.862},
}

abl_df = pd.DataFrame(ablation_data).T
print(abl_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, abl_df.columns, ['Sentiment F1 Macro', 'Condition Accuracy']):
    bars = ax.barh(abl_df.index, abl_df[col], color=['#9e9e9e']*3 + ['#1a73e8'])
    ax.set_xlim(0.84, 0.89)
    ax.set_title(title, fontweight='bold')
    ax.invert_yaxis()
    for bar, val in zip(bars, abl_df[col]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Ablation: Task Weight Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/ablation_weights.png', bbox_inches='tight')
plt.show()

## 5. MC-Dropout Uncertainty

In [ ]:
example_reviews = [
    'This medication completely cured my depression after 6 weeks. Highly recommend.',
    'Caused severe nausea and dizziness. Stopped after two days.',
    'Mixed results. Helped somewhat but the side effects were annoying.',
]

for review in example_reviews:
    enc = tokenizer(review, return_tensors='pt', truncation=True, max_length=256)
    unc = model.mc_uncertainty(
        enc['input_ids'].to(DEVICE),
        enc['attention_mask'].to(DEVICE),
        n_samples=30
    )
    sent_pred = int(unc['sentiment']['mean'].argmax(-1))
    sent_unc  = float(unc['sentiment']['std'].max())
    rating    = float(unc['rating']['mean']) * 9 + 1
    rating_unc = float(unc['rating']['std']) * 9

    labels = ['Negative', 'Neutral', 'Positive']
    print(f'Review: "{review[:60]}..."')
    print(f'  Sentiment: {labels[sent_pred]} (uncertainty: ±{sent_unc:.3f})')
    print(f'  Rating:    {rating:.1f}/10 (±{rating_unc:.2f})')
    print()